# 00 — Probe môi trường Colab

**Chạy notebook này trước mọi thứ khác.** Nó không cài gì nặng, không tốn compute đáng kể
(khoảng 2–3 phút), chỉ đo xem runtime của bạn làm được tới đâu.

**Trước khi chạy:** `Runtime → Change runtime type → GPU → L4` (không có L4 thì T4).

**Chạy xong:** cell số 6 in ra một block JSON — **copy block đó dán cho tôi**, hoặc tải file
`MyDrive/proteinAI/out/00_probe_report.json` rồi gửi thẳng. Nhớ điền tay 2 thứ mà code không đọc được:
**số compute unit còn lại** và **tier Pro/Pro+**.

Nếu cell nào lỗi đỏ: **copy nguyên traceback cho tôi**, đừng tóm tắt — lỗi cũng là dữ liệu, và
`CUDA out of memory` với `ModuleNotFoundError` dẫn tới hai hướng sửa khác nhau hoàn toàn.

---

Bối cảnh: 长江杯 2026, 2 track minibinder — **TL1A** (O95150, 180 aa, trimer) và
**IL-7Rα** (P16871, 219 aa). Hạn submit 2026-11-30 (lập kế hoạch theo mốc an toàn 11-15).
Ta là public team → 20 sequence mỗi track phải vượt cửa sàng **ESMFold2 + ipSAE** mới được đo KD thật.

Kết quả notebook này quyết định: chạy được RFdiffusion/BindCraft tại chỗ, hay phải dựa vào platform sponsor.

In [ ]:
#@title 1. GPU / CPU / RAM / disk
import subprocess, sys, os, shutil, platform, json

REPORT = {}

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=120).stdout.strip()
    except Exception as e:
        return f"ERR {e}"

print("="*70)
print("GPU")
print("="*70)
gpu = sh("nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version,compute_cap --format=csv,noheader")
print(gpu if gpu else "KHÔNG THẤY GPU — hãy đổi Runtime type sang GPU rồi chạy lại!")
REPORT["gpu"] = gpu

print()
print(sh("nvidia-smi"))

print("="*70)
print("CPU / RAM / DISK")
print("="*70)
import multiprocessing
REPORT["cpu_count"] = multiprocessing.cpu_count()
print("CPU cores:", REPORT["cpu_count"])
mem = sh("free -g | head -2")
print(mem)
REPORT["ram"] = mem
du = shutil.disk_usage("/content")
REPORT["disk_free_gb"] = round(du.free/2**30, 1)
print(f"/content free: {REPORT['disk_free_gb']} GB (total {round(du.total/2**30,1)} GB)")

print("="*70)
print("PYTHON / CUDA / TORCH")
print("="*70)
REPORT["python"] = sys.version.split()[0]
print("python", REPORT["python"], "|", platform.platform())
nvcc = sh("nvcc --version | tail -2")
print("nvcc:", nvcc)
REPORT["nvcc"] = nvcc
try:
    import torch
    REPORT["torch"] = torch.__version__
    REPORT["torch_cuda"] = torch.version.cuda
    REPORT["cuda_avail"] = torch.cuda.is_available()
    print(f"torch {torch.__version__} | cuda {torch.version.cuda} | available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        REPORT["gpu_name"] = p.name
        REPORT["gpu_vram_gb"] = round(p.total_memory/2**30, 1)
        print(f"  {p.name}  {REPORT['gpu_vram_gb']} GB  sm_{p.major}{p.minor}")
except ImportError:
    print("torch chưa cài")
    REPORT["torch"] = None
try:
    import jax
    REPORT["jax"] = jax.__version__
    REPORT["jax_devices"] = str(jax.devices())
    print(f"jax {jax.__version__} devices={jax.devices()}")
except ImportError:
    print("jax chưa cài (cần cho ColabDesign/BindCraft/AF2)")
    REPORT["jax"] = None

In [ ]:
#@title 2. Mount Google Drive + tạo cache dir cho weights
# Weights sẽ cache ở Drive để không phải tải lại mỗi session (RFdiffusion ~3GB, ESMFold ~5GB, Boltz ~2GB).
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
ROOT = '/content/drive/MyDrive/proteinAI'
for d in ['weights', 'runs', 'data', 'out']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

du = shutil.disk_usage('/content/drive/MyDrive')
REPORT['drive_free_gb'] = round(du.free/2**30, 1)
REPORT['drive_total_gb'] = round(du.total/2**30, 1)
print(f"Drive: {REPORT['drive_free_gb']} GB trống / {REPORT['drive_total_gb']} GB tổng")
print(f"Đã tạo: {ROOT}/{{weights,runs,data,out}}")

# tốc độ ghi Drive (ảnh hưởng lớn khi cache weights nhiều GB)
import time
t = time.time()
with open(f'{ROOT}/_speedtest.bin','wb') as f:
    f.write(os.urandom(100*2**20))   # 100 MB
dt = time.time()-t
REPORT['drive_write_mbps'] = round(100/dt, 1)
print(f"Drive write: {REPORT['drive_write_mbps']} MB/s")
os.remove(f'{ROOT}/_speedtest.bin')

In [ ]:
#@title 3. Compute budget còn lại + loại runtime
# Colab không có API chính thức cho compute unit; đây là các tín hiệu gián tiếp.
import os, subprocess

print("Runtime hostname:", subprocess.run('hostname',shell=True,capture_output=True,text=True).stdout.strip())
print("COLAB_GPU env:", os.environ.get('COLAB_GPU'))
print("TPU?:", os.environ.get('COLAB_TPU_ADDR', 'không'))
print()
print(">>> SỐ COMPUTE UNIT CÒN LẠI: Colab không cho đọc bằng code.")
print(">>> Bạn mở menu phía trên phải (nút RAM/Disk) hoặc https://colab.research.google.com/signup")
print(">>> rồi gõ tay con số còn lại cho tôi. Rất quan trọng để tôi chia budget GPU-hour.")
print()
print("Ước lượng tiêu thụ (Colab Pro, giá compute unit tại thời điểm viết):")
print("  T4  ~ 1.8 CU/giờ")
print("  L4  ~ 4.8 CU/giờ")
print("  A100~ 12-15 CU/giờ")
print("  => 100 CU trên L4 ~ 20 giờ GPU. Pipeline minibinder 2 target cần khoảng 15-40 giờ GPU.")

In [ ]:
#@title 4. Benchmark GPU thực tế (matmul fp32/fp16/bf16)
# Đo throughput thật, để tôi ước lượng thời gian chạy RFdiffusion / AF2 / ESMFold.
import torch, time
if not torch.cuda.is_available():
    print("Không có GPU, bỏ qua.")
else:
    dev = 'cuda'
    res = {}
    for dt_name, dt in [('fp32', torch.float32), ('fp16', torch.float16), ('bf16', torch.bfloat16)]:
        try:
            n = 4096
            a = torch.randn(n, n, device=dev, dtype=dt)
            b = torch.randn(n, n, device=dev, dtype=dt)
            torch.cuda.synchronize(); t = time.time()
            for _ in range(20):
                c = a @ b
            torch.cuda.synchronize()
            el = time.time() - t
            tflops = 20 * 2 * n**3 / el / 1e12
            res[dt_name] = round(tflops, 1)
            print(f"{dt_name}: {tflops:.1f} TFLOP/s")
            del a, b, c; torch.cuda.empty_cache()
        except Exception as e:
            res[dt_name] = f"ERR {type(e).__name__}"
            print(f"{dt_name}: lỗi {e}")
    REPORT['bench_tflops'] = res

    # VRAM headroom thật: cấp phát tăng dần tới khi OOM
    print()
    print("VRAM cấp phát được tối đa:")
    blocks, got = [], 0
    try:
        while True:
            blocks.append(torch.empty(int(0.5*2**30//4), dtype=torch.float32, device=dev))  # 0.5 GB
            got += 0.5
    except RuntimeError:
        pass
    del blocks; torch.cuda.empty_cache()
    REPORT['vram_allocatable_gb'] = got
    print(f"  ~{got} GB")
    print()
    print("Tham chiếu nhu cầu VRAM:")
    print("  ProteinMPNN           < 4 GB   ✅ chạy đâu cũng được")
    print("  ESMFold (~300 aa)     ~14 GB   ⚠️ T4 16GB rất sát, L4 24GB thoải mái")
    print("  RFdiffusion binder, target ~180-220 aa  ~10-20 GB (tăng theo bình phương chiều dài)")
    print("  AF2 initial-guess, complex ~280 aa      ~8-12 GB")
    print("  BindCraft (AF2 backprop)                ~16-24 GB  ⚠️ T4 thường OOM")

In [ ]:
#@title 5. Kiểm tra mạng: tải được weights từ đâu
# Một số host bị chặn/chậm tuỳ region của Colab. Cần biết trước để không mất 2 tiếng.
import subprocess

HOSTS = {
    'github.com':            'https://github.com',
    'huggingface.co':        'https://huggingface.co',
    'hf CDN (cdn-lfs)':      'https://cdn-lfs.huggingface.co',
    'RFdiffusion weights (IPD)': 'http://files.ipd.uw.edu/pub/RFdiffusion/',
    'RCSB files':            'https://files.rcsb.org',
    'UniProt REST':          'https://rest.uniprot.org',
    'colabfold MSA server':  'https://api.colabfold.com',
    'storage.googleapis':    'https://storage.googleapis.com',
}
net = {}
for name, url in HOSTS.items():
    r = subprocess.run(
        f'curl -sS -o /dev/null -w "%{{http_code}} %{{time_total}}s" --max-time 25 -L "{url}"',
        shell=True, capture_output=True, text=True)
    out = (r.stdout or r.stderr or '').strip()
    net[name] = out
    print(f"{name:30s} {out}")
REPORT['net'] = net

print()
print("Tốc độ tải thực tế (file 100MB từ HF):")
print(subprocess.run(
    'curl -sS -o /dev/null -w "%{speed_download} bytes/s  http=%{http_code}\\n" --max-time 60 -r 0-104857600 -L '
    'https://huggingface.co/facebook/esmfold_v1/resolve/main/pytorch_model.bin',
    shell=True, capture_output=True, text=True).stdout)

In [ ]:
#@title 6. ⬇️ KẾT QUẢ GỬI LẠI CHO CLAUDE — chọn 1 trong 2 cách
import json, os, datetime

REPORT['_notebook']  = '00_probe_env'
REPORT['_timestamp'] = datetime.datetime.now().isoformat(timespec='seconds')
REPORT['_dien_tay']  = {
    'compute_unit_con_lai': '<<ĐIỀN TAY>>',
    'colab_tier':           '<<ĐIỀN TAY: Pro hay Pro+>>',
}

blob = json.dumps(REPORT, indent=2, ensure_ascii=False, default=str)

# ───────── CÁCH 1: copy text dán vào chat ─────────
print("="*72)
print("PASTE CHO CLAUDE — BẮT ĐẦU")
print("="*72)
print(blob)
print("="*72)
print("PASTE CHO CLAUDE — KẾT THÚC")
print("="*72)

# ───────── CÁCH 2: gửi file (tiện hơn, không sợ thiếu chữ) ─────────
try:
    out = '/content/drive/MyDrive/proteinAI/out/00_probe_report.json'
    os.makedirs(os.path.dirname(out), exist_ok=True)
    with open(out, 'w') as f:
        f.write(blob)
    print()
    print(f"✅ Đã ghi file: {out}")
    print("   → Tải file này từ Drive rồi gửi thẳng cho tôi, khỏi copy text.")
except Exception as e:
    print(f"\n(không ghi được ra Drive: {e} — dùng Cách 1)")

print()
print("⚠️  HAI THỨ CODE KHÔNG ĐỌC ĐƯỢC, BẠN ĐIỀN TAY KHI GỬI:")
print("    1. Compute unit còn lại: ______")
print("       (bấm nút RAM/Disk góc trên phải, hoặc vào https://colab.research.google.com/signup)")
print("    2. Colab tier Pro / Pro+: ______")
print()
print("Nếu có cell nào báo lỗi đỏ, copy luôn cả phần lỗi cho tôi — lỗi cũng là thông tin.")